# Step 1A — Load one webpage manually

## Cell 1 — Imports + .env

In [1]:
from pathlib import Path
import os
import requests

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from urllib.parse import urlparse

from langchain_core.documents import Document


# .env is one level above notebooks/
load_dotenv(Path("../.env"))

USER_AGENT = os.getenv(
    "USER_AGENT",
    "Mozilla/5.0 (RealEstateRAG/1.0)"
)

print("Setup complete")

Setup complete


## Cell 2 — Fetch one URL

In [2]:
url = "https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627"

In [3]:
headers = {
    "User-Agent": USER_AGENT,
    "Accept-Language": "en-US,en;q=0.9"
}

response = requests.get(
    url,
    headers=headers,
    timeout=15
)

print("Status code:", response.status_code)
print("HTML characters:", len(response.text))

Status code: 200
HTML characters: 968134


In [4]:
print(response.text[:1000])

<!doctype html>

<html xmlns="http://www.w3.org/1999/xhtml"  lang="en">

<head>
    <title>Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes</title>
    <meta name="description" content="The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.">
    <meta name="keywords" content="Mumbai,housing,Pune">
    
    
           <link rel="canonical" href="https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627" />
        <link rel="preconnect" crossorigin href="https://sb.scorecardresearch.com/" />
    <link rel="preconnect" crossorigin href="https://www.googletagservices.com/" />
    <link rel="preconnect" crossorigin href="https://www.googletagmanager.com/" />  
    <link rel="preload" as="script" href="https:/

# Step 1B — Parse the HTML


## Cell 3 — BeautifulSoup

In [5]:
soup = BeautifulSoup(
    response.text,
    "html.parser"
)

print(soup.title)

<title>Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes</title>


In [6]:
page_title = (
    soup.title.get_text(strip=True)
    if soup.title
    else "Unknown"
)

print(page_title)

Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes


In [7]:
raw_text = soup.get_text(
    separator=" ",
    strip=True
)

print("Extracted characters:", len(raw_text))
print()
print(raw_text[:3000])

Extracted characters: 9077

Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes arrow-down comments printer search bell top-nav right-arrow left-arrow arrow-down Get App Scan to Download Advertisement Live TV Latest Markets Business Economy Research Reports Videos Beeps Personal Finance India World Our Network NDTV हिन्दी World Profit Sports Movies Food Education Lifestyle Health Tech Games Shopping Rajasthan MPCG Marathi Get App for
                                                        Better Experience Get it on Google Play Download on the App Store Follow us
                                                        on facebook twitter instagram linkedin youtube whatsapp Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes facebook twitter WhatsApp home India Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes T

# Step 1C — Do basic cleaning

## Cell 4 — Remove obvious HTML noise

In [8]:
# Make a fresh soup object
clean_soup = BeautifulSoup(
    response.text,
    "html.parser"
)

# Remove elements that usually contain non-content/noise
for tag in clean_soup([
    "script",
    "style",
    "noscript",
    "svg",
    "form",
    "button"
]):
    tag.decompose()


clean_text = clean_soup.get_text(
    separator="\n",
    strip=True
)

print("Before cleaning:", len(raw_text))
print("After cleaning :", len(clean_text))

print()
print(clean_text[:1000])

Before cleaning: 9077
After cleaning : 8779

Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes
Get App
Scan to Download
Advertisement
Live TV
Latest
Markets
Business
Economy
Research Reports
Videos
Beeps
Personal Finance
India
World
Our Network
NDTV
हिन्दी
World
Profit
Sports
Movies
Food
Education
Lifestyle
Health
Tech
Games
Shopping
Rajasthan
MPCG
Marathi
Get App for
                                                        Better Experience
Get it on
Google Play
Download on the
App Store
Follow us
                                                        on
facebook
twitter
instagram
linkedin
youtube
whatsapp
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes
facebook
twitter
WhatsApp
home
India
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes
The Mumbai Metropolitan Region (MMR) and Pune housing markets foll

# Step 1D — Convert it into a LangChain Document

In [9]:
domain = urlparse(url).netloc

doc = Document(
    page_content=clean_text,
    metadata={
        "source": url,
        "title": page_title,
        "domain": domain
    }
)

doc

Document(metadata={'source': 'https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627', 'title': 'Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes', 'domain': 'www.ndtvprofit.com'}, page_content='Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes\nGet App\nScan to Download\nAdvertisement\nLive TV\nLatest\nMarkets\nBusiness\nEconomy\nResearch Reports\nVideos\nBeeps\nPersonal Finance\nIndia\nWorld\nOur Network\nNDTV\nहिन्दी\nWorld\nProfit\nSports\nMovies\nFood\nEducation\nLifestyle\nHealth\nTech\nGames\nShopping\nRajasthan\nMPCG\nMarathi\nGet App for\n                                                        Better Experience\nGet it on\nGoogle Play\nDownload on the\nApp Store\nFollow us\n                                                        on\nfacebook\ntwitter\ninstagram\nlinkedin\nyoutube\nwhatsapp\nMumbai-Pune Housing Market Takes Different Turns in Q

In [10]:
print(type(doc))

<class 'langchain_core.documents.base.Document'>


In [11]:
print(doc.metadata)

{'source': 'https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627', 'title': 'Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes', 'domain': 'www.ndtvprofit.com'}


In [12]:
print(doc.page_content[:1500])

Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes
Get App
Scan to Download
Advertisement
Live TV
Latest
Markets
Business
Economy
Research Reports
Videos
Beeps
Personal Finance
India
World
Our Network
NDTV
हिन्दी
World
Profit
Sports
Movies
Food
Education
Lifestyle
Health
Tech
Games
Shopping
Rajasthan
MPCG
Marathi
Get App for
                                                        Better Experience
Get it on
Google Play
Download on the
App Store
Follow us
                                                        on
facebook
twitter
instagram
linkedin
youtube
whatsapp
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes
facebook
twitter
WhatsApp
home
India
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes
The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-Septe

# Step 2A — Find the useful content container

In [13]:
# Check common containers that may hold the main article

selectors = [
    "article",
    "main",
    '[role="main"]'
]

for selector in selectors:
    element = soup.select_one(selector)

    if element:
        text = element.get_text(
            separator="\n",
            strip=True
        )

        print(
            f"{selector:15} -> "
            f"{len(text):,} characters"
        )
    else:
        print(
            f"{selector:15} -> Not found"
        )

article         -> 3,287 characters
main            -> 4,243 characters
[role="main"]   -> Not found


# Step 2B — Extract only meaningful article blocks

In [14]:
article_element = soup.select_one("article")

if article_element is None:
    article_element = soup.select_one("main")


if article_element is None:
    print("No article/main container found.")

else:

    # Remove obvious noise that may exist inside the article container
    for tag in article_element.select(
        "script, style, noscript, nav, "
        "footer, aside, form, button, svg"
    ):
        tag.decompose()


    # Keep content-oriented elements
    content_blocks = article_element.find_all(
        ["h1", "h2", "h3", "p", "li"]
    )


    lines = []

    for block in content_blocks:

        text = block.get_text(
            " ",
            strip=True
        )

        if text:
            lines.append(text)


    article_text = "\n\n".join(lines)


    print(
        "Article characters:",
        len(article_text)
    )

    print()
    print(article_text[:3000])

Article characters: 3069

Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.

Twitter

WhatsApp

Facebook

Reddit

Email

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier according to an ANAROCK Research report .

MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units launched in Q2 2026.

Pune, meanwhile, recorded approximately 18,730 new housing launch

# Step 2C — Extract useful metadata

In [15]:
def get_meta_content(soup, attr_name, attr_value):
    
    tag = soup.find(
        "meta",
        attrs={attr_name: attr_value}
    )

    if tag:
        return tag.get("content")

    return None


title = (
    get_meta_content(
        soup,
        "property",
        "og:title"
    )
    or page_title
)


description = (
    get_meta_content(
        soup,
        "name",
        "description"
    )
)


published_date = (
    get_meta_content(
        soup,
        "property",
        "article:published_time"
    )
)


author = (
    get_meta_content(
        soup,
        "name",
        "author"
    )
)


metadata = {
    "source": url,
    "domain": urlparse(url).netloc,
    "title": title,
    "description": description,
    "published_date": published_date,
    "author": author
}


metadata

{'source': 'https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627',
 'domain': 'www.ndtvprofit.com',
 'title': 'Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes',
 'description': 'The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.',
 'published_date': None,
 'author': None}

# Step 2D — Build the better Document

In [16]:
article_doc = Document(
    page_content=article_text,
    metadata=metadata
)

print(article_doc.metadata)

print()
print("-" * 80)

print()

print(
    article_doc.page_content[:1500]
)

{'source': 'https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627', 'domain': 'www.ndtvprofit.com', 'title': 'Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes', 'description': 'The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.', 'published_date': None, 'author': None}

--------------------------------------------------------------------------------

Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.

Twitter

WhatsApp

Facebook

Re

# Step 2E — Better block-level cleaning

In [17]:
import re


article_element = soup.select_one("article")

if article_element is None:
    article_element = soup.select_one("main")


# Remove structural noise
for tag in article_element.select(
    "script, style, noscript, nav, footer, aside, "
    "form, button, svg"
):
    tag.decompose()


# Keep mostly article-oriented blocks
content_blocks = article_element.find_all(
    ["h1", "h2", "h3", "p"]
)


skip_exact = {
    "twitter",
    "whatsapp",
    "facebook",
    "reddit",
    "email"
}


skip_prefixes = (
    "also read",
    "also read |",
    "advertisement",
    "read more"
)


clean_blocks = []
seen = set()


for block in content_blocks:

    text = block.get_text(
        " ",
        strip=True
    )

    if not text:
        continue


    normalized = re.sub(
        r"\s+",
        " ",
        text
    ).strip()


    lowered = normalized.lower()


    # Remove social buttons etc.
    if lowered in skip_exact:
        continue


    # Remove related-story/promotional blocks
    if lowered.startswith(skip_prefixes):
        continue


    # Remove exact duplicate blocks
    if normalized in seen:
        continue


    seen.add(normalized)
    clean_blocks.append(normalized)


clean_article_text = "\n\n".join(
    clean_blocks
)


print(
    "Previous article length:",
    len(article_text)
)

print(
    "Cleaner article length :",
    len(clean_article_text)
)

print()
print(clean_article_text[:3000])

Previous article length: 3069
Cleaner article length : 2669

Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier according to an ANAROCK Research report .

MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units launched in Q2 2026.

Pune, meanwhile, recorded approximately 18,730 new housing launched during

# Step 2F — Metadata: why did author and published_date return None?

In [18]:
import json


json_ld_scripts = soup.find_all(
    "script",
    type="application/ld+json"
)


print(
    "JSON-LD blocks found:",
    len(json_ld_scripts)
)

JSON-LD blocks found: 6


## Cell 10

In [19]:
for i, script in enumerate(
    json_ld_scripts,
    start=1
):

    try:

        data = json.loads(
            script.string
        )

        print(
            f"\n--- JSON-LD {i} ---"
        )

        print(data)

    except Exception:
        pass


--- JSON-LD 1 ---
{'@context': 'https://schema.org', '@type': 'NewsMediaOrganization', '@id': 'https://www.ndtvprofit.com/#organization', 'name': 'NDTV Profit', 'url': 'https://www.ndtvprofit.com/', 'description': 'NDTV Profit is a prominent Indian business and financial news brand focused on market news, corporate updates, and economic analysis. It operates as a multi-platform network with a strong presence in digital, television, and social media.', 'sameAs': ['https://twitter.com/NDTVProfitIndia', 'https://www.youtube.com/@NDTVProfitIndia', 'https://facebook.com/NDTVProfit', 'https://www.instagram.com/ndtvprofit', 'https://www.linkedin.com/company/ndtvprofit', 'https://t.me/ndtvprofitnews', 'https://www.whatsapp.com/channel/0029Va9NyUD6RGJPJbs1Z01x', 'https://news.google.com/publications/CAAqBwgKMKr0kAsw9JylAw'], 'logo': {'@type': 'ImageObject', 'url': 'https://www.ndtvprofit.com/image-assets/logos/ndtv_profit_wide.png', 'width': 600, 'height': 60}, 'foundingDate': '2005', 'parentO

## Cell 11 — Extract article metadata from JSON-LD

In [20]:
def find_article_schema(obj):

    if isinstance(obj, dict):

        article_types = {
            "Article",
            "NewsArticle",
            "ReportageNewsArticle"
        }


        obj_type = obj.get("@type")


        if (
            obj_type in article_types
            or (
                isinstance(obj_type, list)
                and any(
                    t in article_types
                    for t in obj_type
                )
            )
        ):
            return obj


        for value in obj.values():

            result = find_article_schema(
                value
            )

            if result:
                return result


    elif isinstance(obj, list):

        for item in obj:

            result = find_article_schema(
                item
            )

            if result:
                return result


    return None

In [21]:
article_schema = None


for script in json_ld_scripts:

    try:

        data = json.loads(
            script.string
        )

        article_schema = (
            find_article_schema(data)
        )


        if article_schema:
            break

    except Exception:
        continue


article_schema

{'@context': 'https://schema.org',
 '@type': 'NewsArticle',
 '@id': 'https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627',
 'inLanguage': 'en',
 'mainEntityOfPage': 'https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627',
 'keywords': 'Mumbai,housing,Pune',
 'url': 'https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627',
 'articleSection': 'India',
 'articleBody': 'The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier according to an ANAROCK Research report.MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 2

In [22]:
if article_schema:

    author_data = article_schema.get(
        "author"
    )


    if isinstance(author_data, dict):

        author_name = author_data.get(
            "name"
        )

    elif isinstance(author_data, list):

        author_name = ", ".join(
            [
                a.get("name", "")
                for a in author_data
                if isinstance(a, dict)
            ]
        )

    else:

        author_name = None


    metadata = {
        "source": url,
        "domain": urlparse(url).netloc,
        "title": article_schema.get(
            "headline",
            page_title
        ),
        "description": article_schema.get(
            "description",
            description
        ),
        "published_date": article_schema.get(
            "datePublished"
        ),
        "modified_date": article_schema.get(
            "dateModified"
        ),
        "author": author_name
    }


else:

    metadata = {
        "source": url,
        "domain": urlparse(url).netloc,
        "title": page_title,
        "description": description,
        "published_date": None,
        "modified_date": None,
        "author": None
    }


metadata

{'source': 'https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627',
 'domain': 'www.ndtvprofit.com',
 'title': 'Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes',
 'description': 'The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.',
 'published_date': '2026-09-29T15:14:34+05:30',
 'modified_date': '2026-09-29T15:46:46+05:30',
 'author': 'Maanya Manchanda'}

In [23]:
clean_doc = Document(
    page_content=clean_article_text,
    metadata=metadata
)


print(
    "Final document characters:",
    len(clean_doc.page_content)
)

print()

print(clean_doc.metadata)

Final document characters: 2669

{'source': 'https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627', 'domain': 'www.ndtvprofit.com', 'title': 'Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes', 'description': 'The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.', 'published_date': '2026-09-29T15:14:34+05:30', 'modified_date': '2026-09-29T15:46:46+05:30', 'author': 'Maanya Manchanda'}


# Step 3 — Chunking

## Cell 12 — Import the splitters

In [24]:
import tiktoken


from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)

print("Splitters imported")




Splitters imported


## Cell 13 — CharacterTextSplitter

In [25]:
character_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=500,
    chunk_overlap=80,
    length_function=len
)

character_chunks = character_splitter.split_documents(
    [clean_doc]
)

print(
    "Number of chunks:",
    len(character_chunks)
)

for i, chunk in enumerate(
    character_chunks,
    start=1
):
    print(
        f"\n{'=' * 70}"
    )
    print(
        f"CHUNK {i} | "
        f"{len(chunk.page_content)} characters"
    )
    print(
        "=" * 70
    )

    print(chunk.page_content)

Number of chunks: 8

CHUNK 1 | 322 characters
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.

CHUNK 2 | 279 characters
The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier according to an ANAROCK Research report .

CHUNK 3 | 479 characters
MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units launched in Q2 2026.

Pune, meanwhile, recorded approximately

## Cell 14 — RecursiveCharacterTextSplitter

In [26]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ],

    length_function=len
)

recursive_chunks = recursive_splitter.split_documents(
    [clean_doc]
)

print(
    "Number of chunks:",
    len(recursive_chunks)
)

for i, chunk in enumerate(
    recursive_chunks,
    start=1
):
    print(
        f"\n{'=' * 70}"
    )
    print(
        f"CHUNK {i} | "
        f"{len(chunk.page_content)} characters"
    )
    print(
        "=" * 70
    )

    print(chunk.page_content)

Number of chunks: 8

CHUNK 1 | 322 characters
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.

CHUNK 2 | 279 characters
The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier according to an ANAROCK Research report .

CHUNK 3 | 479 characters
MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units launched in Q2 2026.

Pune, meanwhile, recorded approximately

## Cell 15 — Compare them numerically

In [27]:
import pandas as pd


comparison = pd.DataFrame({
    "Character Splitter": [
        len(chunk.page_content)
        for chunk in character_chunks
    ],
    "Recursive Splitter": [
        len(chunk.page_content)
        for chunk in recursive_chunks
    ]
})


comparison

,Character Splitter,Recursive Splitter
0,322,322
1,279,279
2,479,479
3,221,221
4,439,439
5,354,354
6,398,398
7,163,163


## Cell 16 — Final tiny cleanup

In [28]:
promo_prefixes = (
    "Essential Business Intelligence",
)

final_blocks = [
    block
    for block in clean_blocks
    if not block.startswith(promo_prefixes)
]

final_article_text = "\n\n".join(final_blocks)

clean_doc = Document(
    page_content=final_article_text,
    metadata=metadata
)

print("Before:", len(clean_article_text))
print("Final :", len(final_article_text))
print()
print(final_article_text[-800:])

Before: 2669
Final : 2504

n Q3. 2025 to Rs.1.55 lakh crore in Q3 2026.

On a quarterly basis, housing sales increased 10 percent from approximately 90,715 units in Q2 2026 to 1,00,220 units in Q3 2026.

The seven cities covered by the report are NCR, MMR, Bengaluru, Pune, Hyderabad, Chennai and Kolkata.

“The upcoming festive season is expected to boost residential demand, building on the momentum seen in Q3,” Puri stated.

“While buyers will likely stay selective due to higher prices and increasing affordability concerns, the combination of festive sentiment, stable borrowing costs and a healthy new launch pipeline should support sales. Demand is resilient, but we expect more measured growth - well-priced, well-located projects that align with the current demand profile will outperform others this festive season.”


## Cell 17

In [29]:
character_250 = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=250,
    chunk_overlap=40,
    length_function=len
)

recursive_250 = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ],
    length_function=len
)


char_chunks_250 = character_250.split_documents(
    [clean_doc]
)

rec_chunks_250 = recursive_250.split_documents(
    [clean_doc]
)


print("Character splitter")
print("------------------")

for i, chunk in enumerate(char_chunks_250, 1):
    print(
        f"Chunk {i}: "
        f"{len(chunk.page_content)} chars"
    )


print("\nRecursive splitter")
print("------------------")

for i, chunk in enumerate(rec_chunks_250, 1):
    print(
        f"Chunk {i}: "
        f"{len(chunk.page_content)} chars"
    )

Created a chunk of size 279, which is longer than the specified 250
Created a chunk of size 258, which is longer than the specified 250
Created a chunk of size 285, which is longer than the specified 250


Character splitter
------------------
Chunk 1: 82 chars
Chunk 2: 238 chars
Chunk 3: 279 chars
Chunk 4: 258 chars
Chunk 5: 219 chars
Chunk 6: 221 chars
Chunk 7: 285 chars
Chunk 8: 152 chars
Chunk 9: 232 chars
Chunk 10: 120 chars
Chunk 11: 398 chars

Recursive splitter
------------------
Chunk 1: 82 chars
Chunk 2: 238 chars
Chunk 3: 247 chars
Chunk 4: 69 chars
Chunk 5: 124 chars
Chunk 6: 134 chars
Chunk 7: 219 chars
Chunk 8: 221 chars
Chunk 9: 183 chars
Chunk 10: 116 chars
Chunk 11: 152 chars
Chunk 12: 232 chars
Chunk 13: 120 chars
Chunk 14: 216 chars
Chunk 15: 182 chars


## Cell 18 — Look at the actual oversized chunks

In [30]:
print("CHARACTER CHUNKS OVER 250")
print("=" * 70)

for i, chunk in enumerate(
    char_chunks_250,
    start=1
):
    if len(chunk.page_content) > 250:
        print(
            f"\nChunk {i} | "
            f"{len(chunk.page_content)} chars"
        )
        print(chunk.page_content)


print("\n\nRECURSIVE CHUNKS OVER 250")
print("=" * 70)

for i, chunk in enumerate(
    rec_chunks_250,
    start=1
):
    if len(chunk.page_content) > 250:
        print(
            f"\nChunk {i} | "
            f"{len(chunk.page_content)} chars"
        )
        print(chunk.page_content)

CHARACTER CHUNKS OVER 250

Chunk 3 | 279 chars
The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier according to an ANAROCK Research report .

Chunk 4 | 258 chars
MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units launched in Q2 2026.

Chunk 7 | 285 chars
The chairman of ANAROCK, Anuj Puri, said in a statement, "Among the top cities, MMR recorded the highest sales of approx. 31,750 units, followed by Bengaluru with approx. 16,670 units. Together, these two cities accounted for 48% of the total sales across the top 7 cities in Q3 2026."

Chunk 11 | 398 chars
“While buyers will likely stay selective due to higher prices an

## Cell 19

In [31]:
def test_chunk_size(
    document,
    chunk_size,
    chunk_overlap
):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " ",
            ""
        ],
        length_function=len
    )

    chunks = splitter.split_documents(
        [document]
    )

    lengths = [
        len(chunk.page_content)
        for chunk in chunks
    ]

    return {
        "chunk_size": chunk_size,
        "overlap": chunk_overlap,
        "num_chunks": len(chunks),
        "min_length": min(lengths),
        "max_length": max(lengths),
        "avg_length": round(
            sum(lengths) / len(lengths),
            1
        ),
        "chunks": chunks
    }

In [32]:
result_300 = test_chunk_size(
    clean_doc,
    chunk_size=300,
    chunk_overlap=50
)

result_700 = test_chunk_size(
    clean_doc,
    chunk_size=700,
    chunk_overlap=100
)

result_1200 = test_chunk_size(
    clean_doc,
    chunk_size=1200,
    chunk_overlap=150
)

In [33]:
chunk_size_comparison = pd.DataFrame([
    {
        k: v
        for k, v in result.items()
        if k != "chunks"
    }
    for result in [
        result_300,
        result_700,
        result_1200
    ]
])

chunk_size_comparison

,chunk_size,overlap,num_chunks,min_length,max_length,avg_length
0,300,50,11,82,285,226.0
1,700,100,5,354,662,499.2
2,1200,150,3,520,1084,874.0


## Cell 20 — Install/import tokenizer 

In [34]:
import tiktoken

print("tiktoken imported")

tiktoken imported


## Cell 21 — Characters, words and tokens

In [35]:
encoding = tiktoken.get_encoding(
    "cl100k_base"
)

sample_text = clean_doc.page_content


character_count = len(sample_text)

word_count = len(
    sample_text.split()
)

token_count = len(
    encoding.encode(sample_text)
)


print("Characters :", character_count)
print("Words      :", word_count)
print("Tokens     :", token_count)

print()
print(
    "Chars/token:",
    round(
        character_count / token_count,
        2
    )
)

Characters : 2504
Words      : 402
Tokens     : 587

Chars/token: 4.27


# Step 5 — See what a token actually is

In [36]:
sample_sentence = (
    "Pune housing launches increased "
    "47 percent quarter-on-quarter."
)


token_ids = encoding.encode(
    sample_sentence
)


print("Text:")
print(sample_sentence)

print("\nToken IDs:")
print(token_ids)

print("\nNumber of tokens:")
print(len(token_ids))

Text:
Pune housing launches increased 47 percent quarter-on-quarter.

Token IDs:
[47, 2957, 11983, 38175, 7319, 220, 2618, 3346, 8502, 10539, 58414, 13]

Number of tokens:
12


In [37]:
for token_id in token_ids:

    token_text = encoding.decode(
        [token_id]
    )

    print(
        token_id,
        repr(token_text)
    )

47 'P'
2957 'une'
11983 ' housing'
38175 ' launches'
7319 ' increased'
220 ' '
2618 '47'
3346 ' percent'
8502 ' quarter'
10539 '-on'
58414 '-quarter'
13 '.'


## Step 6 — Token-based splitting

In [38]:
token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",

    chunk_size=150,
    chunk_overlap=25,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)


token_chunks = token_splitter.split_documents(
    [clean_doc]
)


print(
    "Number of token chunks:",
    len(token_chunks)
)

Number of token chunks: 5


In [39]:
for i, chunk in enumerate(
    token_chunks,
    start=1
):

    text = chunk.page_content

    chars = len(text)

    words = len(
        text.split()
    )

    tokens = len(
        encoding.encode(text)
    )


    print(
        f"\nChunk {i}"
        f" | chars={chars}"
        f" | words={words}"
        f" | tokens={tokens}"
    )

    print("-" * 70)

    print(text)


Chunk 1 | chars=603 | words=92 | tokens=121
----------------------------------------------------------------------
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier according to an ANAROCK Research report .

Chunk 2 | chars=479 | words=80 | tokens=129
----------------------------------------------------------------------
MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 27 percent higher than 

# Step 7 — HTML / structure-aware chunking

## Cell 24 — Import HTML header splitter

In [40]:
from langchain_text_splitters import HTMLHeaderTextSplitter

print("HTML splitter imported")

HTML splitter imported


In [41]:
headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
]

html_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

## Cell 25 — Split the article HTML

In [42]:
article_html = str(article_element)

html_chunks = html_splitter.split_text(
    article_html
)

print("HTML-aware chunks:", len(html_chunks))

HTML-aware chunks: 4


In [43]:
for i, chunk in enumerate(
    html_chunks,
    start=1
):

    print(
        f"\n{'=' * 70}"
    )
    
    print(f"CHUNK {i}")
    
    print("Metadata:")
    print(chunk.metadata)

    print("\nContent:")
    print(chunk.page_content[:1000])


CHUNK 1
Metadata:
{}

Content:
====== Whatsapp Sticky ====== ====== Story Starts ======  
Story Headline Language / Vernacular Links Story description Top Advertisement ====== Post By Social Share Story Content

CHUNK 2
Metadata:
{'Header 1': 'Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes'}

Content:
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

CHUNK 3
Metadata:
{'Header 1': 'Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes', 'Header 2': 'The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.'}

Content:
The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pun

## Step 8 — Combine structural + size-based splitting

In [44]:
final_structured_chunks = []

for section in html_chunks:

    smaller_chunks = token_splitter.split_documents(
        [section]
    )

    final_structured_chunks.extend(
        smaller_chunks
    )


print(
    "Final structured chunks:",
    len(final_structured_chunks)
)

Final structured chunks: 11


In [45]:
for i, chunk in enumerate(
    final_structured_chunks,
    start=1
):

    tokens = len(
        encoding.encode(
            chunk.page_content
        )
    )

    print(
        f"\nChunk {i}"
        f" | tokens={tokens}"
    )

    print(
        "Metadata:",
        chunk.metadata
    )

    print(
        chunk.page_content[:500]
    )


Chunk 1 | tokens=32
Metadata: {}
====== Whatsapp Sticky ====== ====== Story Starts ======  
Story Headline Language / Vernacular Links Story description Top Advertisement ====== Post By Social Share Story Content

Chunk 2 | tokens=24
Metadata: {'Header 1': 'Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes'}
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

Chunk 3 | tokens=44
Metadata: {'Header 1': 'Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes', 'Header 2': 'The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.'}
The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply whi

In [46]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer

# Split our cleaned article into sentences
sentences = re.split(
    r'(?<=[.!?])\s+',
    clean_doc.page_content
)

sentences = [
    s.strip()
    for s in sentences
    if s.strip()
]

# Local embedding model
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# Convert sentences into vectors
embeddings = model.encode(
    sentences,
    normalize_embeddings=True
)

# Compare each sentence with the next sentence
similarities = np.sum(
    embeddings[:-1] * embeddings[1:],
    axis=1
)

for i, score in enumerate(similarities):
    print(f"\nSimilarity: {score:.3f}")
    print("Sentence 1:", sentences[i][:120])
    print("Sentence 2:", sentences[i + 1][:120])

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Similarity: 0.897
Sentence 1: Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

The Mumbai Metropolitan Region (MMR)
Sentence 2: The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quar

Similarity: 0.437
Sentence 1: The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quar
Sentence 2: MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANA

Similarity: 0.516
Sentence 1: MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANA
Sentence 2: The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units launched

Similarity: 0.332
Sentence 1: The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units la

In [47]:
threshold = 0.25

semantic_chunks = []
current_chunk = [sentences[0]]

for i, score in enumerate(similarities):

    if score < threshold:
        semantic_chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i + 1]]

    else:
        current_chunk.append(sentences[i + 1])

# Add final chunk
semantic_chunks.append(" ".join(current_chunk))


print("Number of semantic chunks:", len(semantic_chunks))

for i, chunk in enumerate(semantic_chunks, 1):
    print(f"\n--- CHUNK {i} ---")
    print(chunk)

Number of semantic chunks: 5

--- CHUNK 1 ---
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier. The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier according to an ANAROCK Research report . MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units launched in Q2 2026. Pune, meanwhile, recorded approximately 18,730 new housing launched during the quarter. Whil

# store them in ChromaDB

In [48]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


vectorstore = Chroma.from_documents(
    documents=token_chunks,
    embedding=embedding_model,
    collection_name="real_estate_demo"
)


print("Chunks stored:", vectorstore._collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunks stored: 5


# Similarity Search

In [50]:
question = (
    "How did Pune new housing launches in Q3 2026 "
    "change compared with Q2 2026 and Q3 2025?"
)

results = vectorstore.similarity_search_with_score(
    question,
    k=3
)

for i, (doc, score) in enumerate(results, 1):
    print(f"\n--- RESULT {i} ---")
    print("Distance:", round(score, 4))
    print(doc.page_content)


--- RESULT 1 ---
Distance: 0.4805
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier according to an ANAROCK Research report .

--- RESULT 2 ---
Distance: 0.5929
MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units launched in Q2 2026.

Pune, meanwhile, recorded approximately 18,730 new housing launche

# MMR

In [51]:
mmr_results = vectorstore.max_marginal_relevance_search(
    question,
    k=3,
    fetch_k=5
)

for i, doc in enumerate(mmr_results, 1):
    print(f"\n--- MMR RESULT {i} ---")
    print(doc.page_content)


--- MMR RESULT 1 ---
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier according to an ANAROCK Research report .

--- MMR RESULT 2 ---
MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units launched in Q2 2026.

Pune, meanwhile, recorded approximately 18,730 new housing launched during the quarter. Whil

# Reranking

In [53]:
from sentence_transformers import CrossEncoder

# Retrieve candidate chunks first
candidates = vectorstore.similarity_search(
    question,
    k=5
)

# Load a reranking model
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2"
)

pairs = [
    [question, doc.page_content]
    for doc in candidates
]

scores = reranker.predict(pairs)

ranked = sorted(
    zip(candidates, scores),
    key=lambda x: x[1],
    reverse=True
)

for i, (doc, score) in enumerate(ranked, 1):
    print(f"\n--- RERANKED {i} ---")
    print("Score:", round(float(score), 4))
    print(doc.page_content)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


--- RERANKED 1 ---
Score: 7.7868
MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units launched in Q2 2026.

Pune, meanwhile, recorded approximately 18,730 new housing launched during the quarter. While launches jumped 47 percent from 12,730 units in Q2 2026, they were 3 percent lower than the 19,375 units launched in Q3 2025.

--- RERANKED 2 ---
Score: 4.2579
The total value of housing sales also increased by 2 percent, rising from approximately Rs.1.52 lakh crore in Q3. 2025 to Rs.1.55 lakh crore in Q3 2026.

On a quarterly basis, housing sales increased 10 percent from approximately 90,715 units in Q2 2026 to 1,00,220 units in Q3 2026.

The seven cities covered by the report are NCR, MMR, Bengaluru, Pune, Hyderabad, Chennai and Kolkata.

“The upcoming festive season is expected to boost residential de

# RetrievalQA: manually give the retrieved evidence to Groq and generate the final answer.

In [54]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# Take top 3 reranked chunks
top_docs = [doc for doc, score in ranked[:3]]

context = "\n\n".join(
    doc.page_content
    for doc in top_docs
)

prompt = f"""
Answer the question using ONLY the context below.

If the answer is not available in the context,
say: "The provided sources do not contain enough information."

Question:
{question}

Context:
{context}

Answer:
"""

response = llm.invoke(prompt)

print(response.content)

Pune’s new‑housing launches in Q3 2026 rose sharply from the previous quarter, jumping **47 %** to about **18,730 units** from **12,730 units** in Q2 2026.  However, compared with the same quarter a year earlier, they were **3 % lower**, falling short of the **19,375 units** launched in Q3 2025.


In [55]:
unknown_question = "What was the average home loan interest rate in Pune?"

prompt = f"""
Answer the question using ONLY the context below.

If the answer is not available in the context,
say: "The provided sources do not contain enough information."

Question:
{unknown_question}

Context:
{context}

Answer:
"""

response = llm.invoke(prompt)

print(response.content)

The provided sources do not contain enough information.


# add source citations

In [56]:
top_docs = [doc for doc, score in ranked[:3]]

context_parts = []

for i, doc in enumerate(top_docs, 1):
    source = doc.metadata.get("source", "Unknown source")

    context_parts.append(
        f"[Source {i}]\n"
        f"{doc.page_content}\n"
        f"URL: {source}"
    )

context_with_sources = "\n\n".join(context_parts)

prompt = f"""
Answer the question using ONLY the context below.

Mention the source number(s) used in the answer,
for example [Source 1].

If the answer is not available in the context,
say: "The provided sources do not contain enough information."

Question:
{question}

Context:
{context_with_sources}

Answer:
"""

response = llm.invoke(prompt)

print(response.content)

Pune’s new‑housing launches in Q3 2026 rose sharply from the previous quarter but slipped slightly from the same period a year earlier.  
- Compared with Q2 2026, launches jumped **47 %** (from 12,730 to about 18,730 units).  
- Compared with Q3 2025, launches were **3 % lower** (18,730 versus 19,375 units).  

[Source 1]


In [57]:
# Build unique source IDs
source_ids = {}
source_info = {}
context_parts = []

for doc in top_docs:
    url = doc.metadata.get("source", "Unknown")
    title = doc.metadata.get("title", "Unknown title")

    # Same URL = same source number
    if url not in source_ids:
        source_number = len(source_ids) + 1
        source_ids[url] = source_number

        source_info[source_number] = {
            "title": title,
            "url": url
        }

    source_number = source_ids[url]

    context_parts.append(
        f"[Source {source_number}]\n"
        f"{doc.page_content}"
    )


context_with_sources = "\n\n".join(context_parts)


prompt = f"""
Answer the question using ONLY the context below.

Cite the supporting source using [Source X].

If the answer is not available in the context,
say: "The provided sources do not contain enough information."

Question:
{question}

Context:
{context_with_sources}

Answer:
"""


response = llm.invoke(prompt)


# ----- DISPLAY -----

print("ANSWER")
print("=" * 70)
print(response.content)

print("\nSOURCES")
print("=" * 70)

for number, info in source_info.items():
    print(f"[Source {number}] {info['title']}")
    print(info["url"])
    print()

ANSWER
Pune’s new housing launches in Q3 2026 rose sharply from the previous quarter but slipped slightly from the same period a year earlier.  
- Compared with Q2 2026, launches jumped **47 %** (from 12,730 to about 18,730 units).  
- Compared with Q3 2025, launches were **3 % lower** (18,730 versus 19,375 units).  

[Source 1]

SOURCES
[Source 1] Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes
https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627



### first check which websites actually allow our Python loader to access them—some news/research sites may block requests.

In [58]:
test_urls = [
    url,  # our existing NDTV URL

    "https://www.business-standard.com/finance/personal-finance/foreign-investors-are-back-india-real-estate-draws-record-9-5-bn-in-q3-126092900227_1.html",

    "https://timesofindia.indiatimes.com/real-estate/news/homebuyers-take-note-these-indian-cities-saw-housing-sales-rise-in-q3-heres-what-changed/articleshow/134538706.cms",

    "https://www.mordorintelligence.com/industry-reports/real-estate-industry-in-india",

    "https://www.marknteladvisors.com/research-library/india-real-estate-market-report.html"
]

for test_url in test_urls:
    try:
        res = requests.get(
            test_url,
            headers={"User-Agent": USER_AGENT},
            timeout=15
        )

        print(
            res.status_code,
            len(res.text),
            test_url
        )

    except Exception as e:
        print("ERROR:", test_url, e)

403 534 https://www.ndtvprofit.com/india/mumbai-pune-housing-market-takes-different-turns-in-q3-37-500-vs-18-730-new-homes-12113627
200 210598 https://www.business-standard.com/finance/personal-finance/foreign-investors-are-back-india-real-estate-draws-record-9-5-bn-in-q3-126092900227_1.html
200 289017 https://timesofindia.indiatimes.com/real-estate/news/homebuyers-take-note-these-indian-cities-saw-housing-sales-rise-in-q3-heres-what-changed/articleshow/134538706.cms
200 192048 https://www.mordorintelligence.com/industry-reports/real-estate-industry-in-india
200 115850 https://www.marknteladvisors.com/research-library/india-real-estate-market-report.html


## turn those 4 URLs into LangChain Document objects

In [59]:
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from urllib.parse import urlparse


new_urls = [
    "https://www.business-standard.com/finance/personal-finance/foreign-investors-are-back-india-real-estate-draws-record-9-5-bn-in-q3-126092900227_1.html",
    "https://timesofindia.indiatimes.com/real-estate/news/homebuyers-take-note-these-indian-cities-saw-housing-sales-rise-in-q3-heres-what-changed/articleshow/134538706.cms",
    "https://www.mordorintelligence.com/industry-reports/real-estate-industry-in-india",
    "https://www.marknteladvisors.com/research-library/india-real-estate-market-report.html"
]


def load_url_document(url):

    response = requests.get(
        url,
        headers={"User-Agent": USER_AGENT},
        timeout=15
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # Remove obvious noise
    for tag in soup([
        "script",
        "style",
        "noscript",
        "svg",
        "form",
        "button"
    ]):
        tag.decompose()

    # Prefer article/main content
    container = (
        soup.select_one("article")
        or soup.select_one("main")
        or soup.body
    )

    text = container.get_text(
        separator="\n",
        strip=True
    )

    title = (
        soup.title.get_text(strip=True)
        if soup.title
        else "Unknown"
    )

    return Document(
        page_content=text,
        metadata={
            "source": url,
            "title": title,
            "domain": urlparse(url).netloc
        }
    )


# Keep our already-clean NDTV document
all_docs = [clean_doc]

for u in new_urls:
    try:
        doc = load_url_document(u)
        all_docs.append(doc)

        print(
            doc.metadata["domain"],
            "|",
            len(doc.page_content),
            "characters"
        )

    except Exception as e:
        print("FAILED:", u, e)


print("\nTotal documents:", len(all_docs))

www.business-standard.com | 6164 characters
timesofindia.indiatimes.com | 15309 characters
www.mordorintelligence.com | 28194 characters
www.marknteladvisors.com | 35418 characters

Total documents: 5


In [60]:
for i, doc in enumerate(all_docs, 1):

    print("\n" + "=" * 80)
    print(f"DOCUMENT {i}")
    print(doc.metadata["domain"])
    print(doc.metadata["title"])
    print("=" * 80)

    print(doc.page_content[:1200])


DOCUMENT 1
www.ndtvprofit.com
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier.

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in new housing supply while Pune saw launches decline from a year earlier according to an ANAROCK Research report .

MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units launched in Q2 2026.

Pune, meanwhile, reco

## improve our loader

In [61]:
import json
import html


def get_jsonld_article_body(soup):
    scripts = soup.find_all(
        "script",
        type="application/ld+json"
    )

    for script in scripts:
        try:
            data = json.loads(script.string)

            items = data if isinstance(data, list) else [data]

            for item in items:
                if not isinstance(item, dict):
                    continue

                # Sometimes JSON-LD contains @graph
                candidates = item.get("@graph", [item])

                if not isinstance(candidates, list):
                    candidates = [candidates]

                for candidate in candidates:
                    if isinstance(candidate, dict):
                        body = candidate.get("articleBody")

                        if body:
                            return html.unescape(body)

        except Exception:
            continue

    return None


def load_url_document_v2(url):

    response = requests.get(
        url,
        headers={"User-Agent": USER_AGENT},
        timeout=15
    )

    response.raise_for_status()

    # Helps fix encoding issues such as TOI mojibake
    response.encoding = response.apparent_encoding

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    title = (
        soup.title.get_text(strip=True)
        if soup.title
        else "Unknown"
    )

    # First choice: structured article content
    text = get_jsonld_article_body(soup)

    # Fallback for pages without articleBody
    if not text:

        for tag in soup([
            "script",
            "style",
            "noscript",
            "svg",
            "form",
            "button",
            "nav",
            "footer"
        ]):
            tag.decompose()

        container = (
            soup.select_one("article")
            or soup.select_one("main")
            or soup.body
        )

        text = container.get_text(
            separator="\n",
            strip=True
        )

    return Document(
        page_content=text,
        metadata={
            "source": url,
            "title": title,
            "domain": urlparse(url).netloc
        }
    )

In [62]:
cleaned_new_docs = []

for u in new_urls:
    doc = load_url_document_v2(u)
    cleaned_new_docs.append(doc)

    print(
        doc.metadata["domain"],
        "|",
        len(doc.page_content),
        "characters"
    )

www.business-standard.com | 4573 characters
timesofindia.indiatimes.com | 3816 characters
www.mordorintelligence.com | 28194 characters
www.marknteladvisors.com | 35371 characters


In [63]:
for doc in cleaned_new_docs:

    print("\n" + "=" * 80)
    print(doc.metadata["domain"])
    print("=" * 80)

    print(doc.page_content[:1000])


www.business-standard.com
India's real estate market attracted a record $9.5 billion in equity capital in the July-September quarter of 2026, more than twice the investment recorded in the same quarter last year, as foreign and institutional investors increased their exposure to the sector. According to CBRE's India Market Monitor – Investments Q3 2026, the quarter's inflows were more than double the $4.4 billion recorded in Q3 2025 and significantly higher than the $3.8 billion invested in Q2 2026. The $9.5-billion quarterly inflow was the highest on record, according to CBRE. ""This is a landmark quarter for India's real estate capital markets," said Anshuman Magazine, Chairman and CEO, India, South-East Asia, Middle East and Africa, CBRE. "Global investors have returned with conviction, and institutional capital is now flowing well beyond offices and land into data centres. It reflects how deep and diverse India's real estate market has become, and we expect this confidence to carr

# Combine all 5 documents

In [64]:
# Combine all 5 documents
multi_docs = [clean_doc] + cleaned_new_docs

# Split all documents into token-based chunks
multi_chunks = token_splitter.split_documents(multi_docs)

print("Documents:", len(multi_docs))
print("Total chunks:", len(multi_chunks))

# See how many chunks came from each website
from collections import Counter

source_counts = Counter(
    chunk.metadata["domain"]
    for chunk in multi_chunks
)

for source, count in source_counts.items():
    print(source, "->", count, "chunks")

Documents: 5
Total chunks: 152
www.ndtvprofit.com -> 5 chunks
www.business-standard.com -> 9 chunks
timesofindia.indiatimes.com -> 8 chunks
www.mordorintelligence.com -> 66 chunks
www.marknteladvisors.com -> 64 chunks


# reate the multi-source ChromaDB and search it

In [65]:
multi_vectorstore = Chroma.from_documents(
    documents=multi_chunks,
    embedding=embedding_model,
    collection_name="real_estate_multi_source"
)

print("Chunks stored:", multi_vectorstore._collection.count())


question_multi = "What were the major trends in India's real estate market in Q3 2026?"

results = multi_vectorstore.similarity_search_with_score(
    question_multi,
    k=8
)

for i, (doc, score) in enumerate(results, 1):

    print(f"\n--- RESULT {i} ---")
    print("Domain:", doc.metadata["domain"])
    print("Distance:", round(score, 4))
    print(doc.page_content[:500])

Chunks stored: 152

--- RESULT 1 ---
Domain: www.marknteladvisors.com
Distance: 0.4463
India Real Estate Market Research Report: Trends & Forecast (2026-2032)
By Property Type (Residential Real Estate, Commercial Real Estate, Industrial Real Estate, Land), By Business Model (Sales / Ownership, Rental / Leasing, Real Estate Investment Tru
...
sts (REITs), Residential, Commercial / Industrial), By Price Range (Affordable Housing, Mid-Segment Housing, Premium Housing, Luxury, Ultra-Luxury), By Construction Type (New Construction, Resale Properties), By Development Stage (Under Constr

--- RESULT 2 ---
Domain: www.mordorintelligence.com
Distance: 0.4927
The report provides a comprehensive background analysis of the market, covering the current market trends, restraints, technological updates, and detailed information on various segments and the competitive landscape of the industry.
The real estate industry in India is segmented by property type (residential, office, retail, hospitality, a

# metadata filtering

In [66]:
news_domains = [
    "www.ndtvprofit.com",
    "www.business-standard.com",
    "timesofindia.indiatimes.com"
]

filtered_results = multi_vectorstore.similarity_search_with_score(
    question_multi,
    k=8,
    filter={
        "domain": {
            "$in": news_domains
        }
    }
)

for i, (doc, score) in enumerate(filtered_results, 1):
    print(f"\n--- RESULT {i} ---")
    print("Domain:", doc.metadata["domain"])
    print("Distance:", round(score, 4))
    print(doc.page_content[:400])


--- RESULT 1 ---
Domain: www.business-standard.com
Distance: 0.5262
. It reflects how deep and diverse India's real estate market has become, and we expect this confidence to carry through the rest of the year." The surge also means that India's real estate sector has already attracted $18.6 billion during the first nine months of 2026, nearly twice the corresponding period of 2025 and above the $14.2 billion recorded during the whole of 2025.  Where is the money 

--- RESULT 2 ---
Domain: www.business-standard.com
Distance: 0.5548
India's real estate market attracted a record $9.5 billion in equity capital in the July-September quarter of 2026, more than twice the investment recorded in the same quarter last year, as foreign and institutional investors increased their exposure to the sector. According to CBRE's India Market Monitor – Investments Q3 2026, the quarter's inflows were more than double the $4.4 billion recorded 

--- RESULT 3 ---
Domain: www.business-standard.com
Distance

# source balancing

In [67]:
balanced_results = []
domain_count = {}

for doc, score in filtered_results:

    domain = doc.metadata["domain"]

    if domain_count.get(domain, 0) < 2:
        balanced_results.append((doc, score))
        domain_count[domain] = domain_count.get(domain, 0) + 1


for i, (doc, score) in enumerate(balanced_results, 1):

    print(f"\n--- RESULT {i} ---")
    print("Domain:", doc.metadata["domain"])
    print("Distance:", round(score, 4))
    print(doc.page_content[:400])


--- RESULT 1 ---
Domain: www.business-standard.com
Distance: 0.5262
. It reflects how deep and diverse India's real estate market has become, and we expect this confidence to carry through the rest of the year." The surge also means that India's real estate sector has already attracted $18.6 billion during the first nine months of 2026, nearly twice the corresponding period of 2025 and above the $14.2 billion recorded during the whole of 2025.  Where is the money 

--- RESULT 2 ---
Domain: www.business-standard.com
Distance: 0.5548
India's real estate market attracted a record $9.5 billion in equity capital in the July-September quarter of 2026, more than twice the investment recorded in the same quarter last year, as foreign and institutional investors increased their exposure to the sector. According to CBRE's India Market Monitor – Investments Q3 2026, the quarter's inflows were more than double the $4.4 billion recorded 

--- RESULT 3 ---
Domain: timesofindia.indiatimes.com
Distan

# rerank these 6 chunks

In [68]:
pairs = [
    [question_multi, doc.page_content]
    for doc, score in balanced_results
]

rerank_scores = reranker.predict(pairs)

multi_ranked = sorted(
    [
        (doc, original_score, rerank_score)
        for (doc, original_score), rerank_score
        in zip(balanced_results, rerank_scores)
    ],
    key=lambda x: x[2],
    reverse=True
)

for i, (doc, distance, rerank_score) in enumerate(multi_ranked, 1):

    print(f"\n--- RERANKED {i} ---")
    print("Domain:", doc.metadata["domain"])
    print("Rerank score:", round(float(rerank_score), 4))
    print(doc.page_content[:400])


--- RERANKED 1 ---
Domain: www.business-standard.com
Rerank score: 6.862
India's real estate market attracted a record $9.5 billion in equity capital in the July-September quarter of 2026, more than twice the investment recorded in the same quarter last year, as foreign and institutional investors increased their exposure to the sector. According to CBRE's India Market Monitor – Investments Q3 2026, the quarter's inflows were more than double the $4.4 billion recorded 

--- RERANKED 2 ---
Domain: timesofindia.indiatimes.com
Rerank score: 5.9141
. Bengaluru, Hyderabad and Mumbai’s wider region saw more homes change hands in the July-September quarter, while sales fell in Pune, Delhi-NCR, Chennai and Kolkata.Across the seven major cities, residential sales rose 3% annually to 1,00,220 units in Q3 2026, compared with 97,080 units in the same quarter last year, according to real estate consultant Anarock.In value terms, housing sales rose 2%

--- RERANKED 3 ---
Domain: www.business-standa

# real multi-source RAG answer.
## Goal
### Use only the top reranked chunks, answer the question, and cite the source numbers.

In [69]:
# Take top 4 reranked chunks
top_multi_docs = [item[0] for item in multi_ranked[:4]]

source_ids = {}
source_info = {}
context_parts = []

for doc in top_multi_docs:
    url = doc.metadata["source"]
    title = doc.metadata["title"]

    if url not in source_ids:
        source_num = len(source_ids) + 1
        source_ids[url] = source_num

        source_info[source_num] = {
            "title": title,
            "url": url
        }

    source_num = source_ids[url]

    context_parts.append(
        f"[Source {source_num}]\n"
        f"{doc.page_content}"
    )

context_multi = "\n\n".join(context_parts)

prompt = f"""
Answer the question using ONLY the context below.

Summarize the major trends clearly and concisely.
Cite the supporting source using [Source X].

If the context is insufficient, say so.

Question:
{question_multi}

Context:
{context_multi}

Answer:
"""

response = llm.invoke(prompt)

print("ANSWER")
print("=" * 70)
print(response.content)

print("\nSOURCES")
print("=" * 70)

for num, info in source_info.items():
    print(f"[Source {num}] {info['title']}")
    print(info["url"])
    print()

ANSWER
**Major trends in India’s real‑estate market – Q3 2026**

| Trend | Key figures | Source |
|-------|-------------|--------|
| **Record equity‑capital inflows** | $9.5 billion, more than double Q3 2025 ($4.4 billion) and higher than Q2 2026 ($3.8 billion). | [Source 1] |
| **Segment‑wide investment** | Data‑centres, built‑up office assets and land/development sites together accounted for ~91 % of Q3 2026 inflows. | [Source 1] |
| **Residential sales volume** | 100,220 units sold across seven major cities – a 3 % annual rise from 97,080 units in Q3 2025; a 10 % quarterly jump from 90,715 units in Q2 2026. | [Source 2], [Source 3] |
| **Residential sales value** | Rs 1.55 lakh crore, up 2 % from Rs 1.52 lakh crore in Q3 2025. | [Source 2], [Source 3] |
| **Regional variation** | Bengaluru, Hyderabad and Mumbai saw higher turnover; Pune, Delhi‑NCR, Chennai and Kolkata experienced declines. | [Source 2] |
| **Year‑to‑date momentum** | $18.6 billion attracted in the first nine months 

# verification step: after generating the answer, another pass checks every claim against the retrieved context.

In [73]:
draft_answer = response.content

verification_prompt = f"""
You are verifying a RAG answer.

Use ONLY the provided context.

Check every factual claim, number, percentage and comparison
in the draft answer.

Rules:
- Correct any claim not directly supported by the context.
- Remove unsupported claims.
- Do not add outside knowledge.
- Preserve source citations like [Source 1].
- Return only the corrected final answer.

Context:
{context_multi}

Draft Answer:
{draft_answer}
"""

verified_response = llm.invoke(verification_prompt)

print(verified_response.content)

**Major trends in India’s real‑estate market – Q3 2026**

| Trend | Key figures | Source |
|-------|-------------|--------|
| **Record equity‑capital inflows** | $9.5 billion, more than double Q3 2025 ($4.4 billion) and higher than Q2 2026 ($3.8 billion). | [Source 1] |
| **Segment‑wide investment** | Data‑centres, built‑up office assets and land/development sites together accounted for ~91 % of Q3 2026 inflows. | [Source 1] |
| **Residential sales volume** | 100,220 units sold across seven major cities – a 3 % annual rise from 97,080 units in Q3 2025; a 10 % quarterly jump from 90,715 units in Q2 2026. | [Source 2], [Source 3] |
| **Residential sales value** | Rs 1.55 lakh crore, up 2 % from Rs 1.52 lakh crore in Q3 2025. | [Source 2], [Source 3] |
| **Regional variation** | Bengaluru, Hyderabad and Mumbai saw higher turnover; Pune, Delhi‑NCR, Chennai and Kolkata experienced declines. | [Source 2] |
| **Year‑to‑date momentum** | $18.6 billion attracted in the first nine months of 2026

In [76]:
claim = (
    "The first nine months of 2026 already doubled "
    "2025's total real-estate inflows."
)

check_prompt = f"""
Using ONLY the context below, verify this claim.

Context:
{context_multi}

Claim:
{claim}

Return exactly:

SUPPORTED: Yes or No
REASON: brief explanation
CORRECTED CLAIM: corrected version if needed
"""

check = llm.invoke(check_prompt)

print(check.content)

SUPPORTED: No  
REASON: The data shows $18.6 billion in the first nine months of 2026 versus $14.2 billion in 2025, which is about 1.3 ×, not a full 2 ×.  
CORRECTED CLAIM: The first nine months of 2026 attracted $18.6 billion in real‑estate inflows, nearly twice the $14.2 billion recorded in the same period of 2025.


In [77]:
check_prompt = f"""
Use ONLY the context below.

Claim:
{claim}

First identify the exact evidence from the context.

Return exactly:

EVIDENCE:
<the relevant sentence(s) from the context>

SUPPORTED:
Yes or No

REASON:
<brief explanation>

CORRECTED CLAIM:
<corrected claim if unsupported>

Context:
{context_multi}
"""

check = llm.invoke(check_prompt)

print(check.content)

EVIDENCE:
"India's real estate sector has already attracted $18.6 billion during the first nine months of 2026, nearly twice the corresponding period of 2025 and above the $14.2 billion recorded during the whole of 2025."

SUPPORTED:
Yes

REASON:
The evidence shows that the first nine months of 2026 attracted $18.6 billion, which is nearly double the $14.2 billion recorded for the entire year of 2025, confirming the claim.

CORRECTED CLAIM:
<none>


# query transformation / multi-query retrieval

In [78]:
query_prompt = f"""
Rewrite the user question into 3 short search queries.

Each query should focus on a different aspect.
Return ONLY the 3 queries, one per line.

User question:
{question_multi}
"""

query_response = llm.invoke(query_prompt)

queries = [
    q.strip("- ").strip()
    for q in query_response.content.splitlines()
    if q.strip()
]

print("Generated queries:")
for q in queries:
    print("-", q)


# Retrieve chunks for every query
multi_query_docs = []

for q in queries:
    docs = multi_vectorstore.similarity_search(
        q,
        k=4,
        filter={
            "domain": {
                "$in": news_domains
            }
        }
    )

    multi_query_docs.extend(docs)


# Remove duplicate chunks
unique_docs = {}

for doc in multi_query_docs:
    key = (
        doc.metadata["source"],
        doc.page_content
    )

    unique_docs[key] = doc


unique_docs = list(unique_docs.values())

print("\nUnique retrieved chunks:", len(unique_docs))

for i, doc in enumerate(unique_docs, 1):
    print(
        f"{i}. {doc.metadata['domain']} | "
        f"{doc.page_content[:120]}..."
    )

Generated queries:
- India real estate Q3 2026 trends
- Q3 2026 India residential property trends
- Q3 2026 India commercial real estate trends

Unique retrieved chunks: 6
1. www.business-standard.com | . It reflects how deep and diverse India's real estate market has become, and we expect this confidence to carry through...
2. www.business-standard.com | still remaining. For context, CBRE had reported that India's real estate sector attracted $14.3 billion in 2025, an incr...
3. www.business-standard.com | India's real estate market attracted a record $9.5 billion in equity capital in the July-September quarter of 2026, more...
4. timesofindia.indiatimes.com | . Bengaluru, Hyderabad and Mumbai’s wider region saw more homes change hands in the July-September quarter, while sales ...
5. www.ndtvprofit.com | The total value of housing sales also increased by 2 percent, rising from approximately Rs.1.52 lakh crore in Q3. 2025 t...
6. timesofindia.indiatimes.com | Buying a home in India’s 

# Multi-query finds a wider candidate set → reranker decides which chunks matter most.

In [79]:
pairs = [
    [question_multi, doc.page_content]
    for doc in unique_docs
]

scores = reranker.predict(pairs)

multi_query_ranked = sorted(
    zip(unique_docs, scores),
    key=lambda x: x[1],
    reverse=True
)

for i, (doc, score) in enumerate(multi_query_ranked, 1):
    print(f"\n--- RANK {i} ---")
    print("Domain:", doc.metadata["domain"])
    print("Score:", round(float(score), 4))
    print(doc.page_content[:350])


--- RANK 1 ---
Domain: www.business-standard.com
Score: 6.862
India's real estate market attracted a record $9.5 billion in equity capital in the July-September quarter of 2026, more than twice the investment recorded in the same quarter last year, as foreign and institutional investors increased their exposure to the sector. According to CBRE's India Market Monitor – Investments Q3 2026, the quarter's inflow

--- RANK 2 ---
Domain: timesofindia.indiatimes.com
Score: 5.9141
. Bengaluru, Hyderabad and Mumbai’s wider region saw more homes change hands in the July-September quarter, while sales fell in Pune, Delhi-NCR, Chennai and Kolkata.Across the seven major cities, residential sales rose 3% annually to 1,00,220 units in Q3 2026, compared with 97,080 units in the same quarter last year, according to real estate consul

--- RANK 3 ---
Domain: www.business-standard.com
Score: 5.1731
. It reflects how deep and diverse India's real estate market has become, and we expect this confidence t

# Multi-query → retrieve → deduplicate → rerank → source-balance → LLM

In [80]:
final_docs = []
source_count = {}

for doc, score in multi_query_ranked:

    domain = doc.metadata["domain"]

    if source_count.get(domain, 0) < 2:
        final_docs.append(doc)
        source_count[domain] = source_count.get(domain, 0) + 1

    if len(final_docs) == 5:
        break


for i, doc in enumerate(final_docs, 1):
    print(
        f"{i}. {doc.metadata['domain']} | "
        f"{doc.page_content[:180]}..."
    )

1. www.business-standard.com | India's real estate market attracted a record $9.5 billion in equity capital in the July-September quarter of 2026, more than twice the investment recorded in the same quarter last...
2. timesofindia.indiatimes.com | . Bengaluru, Hyderabad and Mumbai’s wider region saw more homes change hands in the July-September quarter, while sales fell in Pune, Delhi-NCR, Chennai and Kolkata.Across the seve...
3. www.business-standard.com | . It reflects how deep and diverse India's real estate market has become, and we expect this confidence to carry through the rest of the year." The surge also means that India's re...
4. www.ndtvprofit.com | The total value of housing sales also increased by 2 percent, rising from approximately Rs.1.52 lakh crore in Q3. 2025 to Rs.1.55 lakh crore in Q3 2026.

On a quarterly basis, hous...
5. timesofindia.indiatimes.com | Buying a home in India’s top cities? The latest sales numbers show that the housing market is moving at differen

# Question → multi-query → vector search → deduplicate → rerank → source balance → LLM → cited answer

In [81]:
source_ids = {}
source_info = {}
context_parts = []

for doc in final_docs:

    url = doc.metadata["source"]
    title = doc.metadata["title"]

    if url not in source_ids:
        source_num = len(source_ids) + 1
        source_ids[url] = source_num

        source_info[source_num] = {
            "title": title,
            "url": url
        }

    source_num = source_ids[url]

    context_parts.append(
        f"[Source {source_num}]\n"
        f"{doc.page_content}"
    )


final_context = "\n\n".join(context_parts)


prompt = f"""
Answer the question using ONLY the provided context.

Rules:
- Give a concise summary of the major trends.
- Every factual point must include its supporting [Source X].
- Do not combine different time periods incorrectly.
- Do not add outside knowledge.
- If something is not supported, do not mention it.

Question:
{question_multi}

Context:
{final_context}

Answer:
"""


final_response = llm.invoke(prompt)


print("ANSWER")
print("=" * 70)
print(final_response.content)

print("\nSOURCES")
print("=" * 70)

for num, info in source_info.items():
    print(f"[Source {num}] {info['title']}")
    print(info["url"])
    print()

ANSWER
**Major trends in India’s real‑estate market – Q3 2026**

| Trend | Supporting evidence |
|-------|---------------------|
| **Record equity‑capital inflows** – $9.5 billion, more than twice the $4.4 billion attracted in Q3 2025 and higher than the $3.8 billion in Q2 2026. | [Source 1] |
| **Strong residential sales growth** – 3 % annual rise to 100,220 units (vs. 97,080 units in Q3 2025) and a 10 % quarterly jump from 90,715 units in Q2 2026. | [Source 2] ; [Source 3] |
| **Value of housing sales up 2 % YoY** – Rs 1.55 lakh crore in Q3 2026 versus Rs 1.52 lakh crore in Q3 2025. | [Source 2] ; [Source 3] |
| **Geographic variation** – Bengaluru, Hyderabad and the Mumbai Metropolitan Region saw increased home‑sales activity, while Pune, Delhi‑NCR, Chennai and Kolkata experienced declines. | [Source 2] |
| **Investment spread across segments** – Data‑centres, built‑up office assets and land/development sites accounted for nearly 91 % of Q3 2026 inflows. | [Source 1] |
| **Seasonal 

# RAG evaluation

In [82]:
test_questions = [
    "How did Pune housing launches change in Q3 2026?",
    "How much capital entered India's real estate market in Q3 2026?",
    "Which cities saw housing sales increase in Q3 2026?",
    "What is the expected size of India's real estate market by 2031?"
]

for question in test_questions:

    results = multi_vectorstore.similarity_search(
        question,
        k=3
    )

    print("\n" + "=" * 70)
    print("QUESTION:", question)

    for i, doc in enumerate(results, 1):
        print(
            f"\n{i}. {doc.metadata['domain']}"
        )
        print(doc.page_content[:250])


QUESTION: How did Pune housing launches change in Q3 2026?

1. www.ndtvprofit.com
Mumbai-Pune Housing Market Takes Different Turns in Q3: 37,500 vs 18,730 New Homes

The Mumbai Metropolitan Region (MMR) and Pune housing markets followed different trajectories in the July-September quarter, with MMR recording a sharp annual rise in

2. www.ndtvprofit.com
MMR recorded approximately 37,500 new house launches in Q3 2026, the highest among the seven major cities tracked by ANAROCK. The figure was 27 percent higher than the 29,565 units launched in Q3 2025 and 9 percent above the 34,555 units launched in 

3. www.ndtvprofit.com
The total value of housing sales also increased by 2 percent, rising from approximately Rs.1.52 lakh crore in Q3. 2025 to Rs.1.55 lakh crore in Q3 2026.

On a quarterly basis, housing sales increased 10 percent from approximately 90,715 units in Q2 2

QUESTION: How much capital entered India's real estate market in Q3 2026?

1. www.business-standard.com
India's real 

## Recall@K → did we retrieve the needed evidence?
## MRR → how high was the first correct evidence?

# automate Recall@K and MRR

In [83]:
evaluation_set = [
    {
        "question": "How did Pune housing launches change in Q3 2026?",
        "expected": ["18,730", "47 percent"]
    },
    {
        "question": "How much capital entered India's real estate market in Q3 2026?",
        "expected": ["9.5 billion"]
    },
    {
        "question": "Which cities saw housing sales increase in Q3 2026?",
        "expected": ["Bengaluru", "Hyderabad"]
    },
    {
        "question": "What is the expected size of India's real estate market by 2031?",
        "expected": ["926.56 Billion"]
    }
]


def evaluate_retrieval(vectorstore, dataset, k=3):

    reciprocal_ranks = []
    hits = 0

    for item in dataset:

        docs = vectorstore.similarity_search(
            item["question"],
            k=k
        )

        first_correct_rank = None

        for rank, doc in enumerate(docs, 1):

            text = doc.page_content.lower()

            if all(
                keyword.lower() in text
                for keyword in item["expected"]
            ):
                first_correct_rank = rank
                break


        if first_correct_rank:
            hits += 1
            reciprocal_ranks.append(
                1 / first_correct_rank
            )
        else:
            reciprocal_ranks.append(0)


        print(
            item["question"],
            "-> rank:",
            first_correct_rank
        )


    recall_at_k = hits / len(dataset)

    mrr = sum(reciprocal_ranks) / len(dataset)


    print("\nRecall@", k, ":", round(recall_at_k, 3))
    print("MRR:", round(mrr, 3))


evaluate_retrieval(
    multi_vectorstore,
    evaluation_set,
    k=3
)

How did Pune housing launches change in Q3 2026? -> rank: 2
How much capital entered India's real estate market in Q3 2026? -> rank: 1
Which cities saw housing sales increase in Q3 2026? -> rank: 1
What is the expected size of India's real estate market by 2031? -> rank: 1

Recall@ 3 : 1.0
MRR: 0.875


In [84]:
eval_question = "How much capital entered India's real estate market in Q3 2026?"

eval_docs = multi_vectorstore.similarity_search(
    eval_question,
    k=3
)

eval_context = "\n\n".join(
    doc.page_content
    for doc in eval_docs
)

eval_prompt = f"""
Answer using ONLY the context.

Question:
{eval_question}

Context:
{eval_context}

Answer:
"""

eval_response = llm.invoke(eval_prompt)

print("ANSWER:")
print(eval_response.content)

print("\nEXPECTED FACT:")
print("$9.5 billion in Q3 2026")

ANSWER:
$9.5 billion.

EXPECTED FACT:
$9.5 billion in Q3 2026


In [86]:
import re
import unicodedata

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()


def simple_answer_eval(answer, expected_fact, context):

    answer_n = normalize_text(answer)
    expected_n = normalize_text(expected_fact)
    context_n = normalize_text(context)

    return {
        "correctness": expected_n in answer_n,
        "faithfulness": expected_n in context_n,
        "relevance": len(answer_n) > 0
    }


result = simple_answer_eval(
    eval_response.content,
    "$9.5 billion",
    eval_context
)

print(result)

{'correctness': True, 'faithfulness': True, 'relevance': True}


# Question → retrieval → context → LLM → answer → evaluation

In [88]:
test_set = [
    {
        "question": "How did Pune housing launches change in Q3 2026?",
        "expected": "18,730"
    },
    {
        "question": "How much capital entered India's real estate market in Q3 2026?",
        "expected": "$9.5 billion"
    },
    {
        "question": "Which cities saw housing sales increase in Q3 2026?",
        "expected": "Bengaluru"
    },
    {
        "question": "What is the expected size of India's real estate market by 2031?",
        "expected": "926.56"
    }
]


correct = 0

for item in test_set:

    docs = multi_vectorstore.similarity_search(
        item["question"],
        k=3
    )

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )

    prompt = f"""
Answer using ONLY the context.

Question:
{item["question"]}

Context:
{context}

Answer:
"""

    response = llm.invoke(prompt)

    answer = response.content

    passed = (
        normalize_text(item["expected"])
        in normalize_text(answer)
    )

    if passed:
        correct += 1

    print("\nQuestion:", item["question"])
    print("Answer:", answer)
    print("PASS:", passed)


print(
    "\nOverall accuracy:",
    f"{correct}/{len(test_set)}",
    "=",
    round(correct / len(test_set), 2)
)


Question: How did Pune housing launches change in Q3 2026?
Answer: Pune’s new‑home launches rose sharply in Q3 2026, jumping 47 % from the 12,730 units launched in Q2 2026 to about 18,730 units. However, this figure was still 3 % lower than the 19,375 units that had been launched in Q3 2025.
PASS: True

Question: How much capital entered India's real estate market in Q3 2026?
Answer: $9.5 billion.
PASS: True

Question: Which cities saw housing sales increase in Q3 2026?
Answer: Bengaluru, Hyderabad, and the Mumbai Metropolitan Region (MMR) saw housing sales increase in Q3 2026.
PASS: True

Question: What is the expected size of India's real estate market by 2031?
Answer: The India real estate market is projected to reach **USD 926.56 billion by 2031**.
PASS: True

Overall accuracy: 4/4 = 1.0


### At this point, we’ve covered the core notebook concepts well enough:
- document loading and cleaning
- metadata
- recursive / token / HTML / semantic chunking
- embeddings
- ChromaDB
- similarity retrieval
- Top-K
- metadata filtering
- MMR
- reranking
- multi-query retrieval
- source balancing
- grounded generation
- citations
- retrieval evaluation
- answer evaluation